In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# =========================
# DEVICE CONFIGURATION
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing Device: {device}\n")

# =========================
# DATA TRANSFORMS
# =========================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

# =========================
# LOAD DATASET
# =========================
dataset = datasets.ImageFolder("dataset", transform=transform)

class_names = dataset.classes
num_classes = len(class_names)

print("Classes Found:")
for idx, name in enumerate(class_names):
    print(f"{idx}: {name}")

# =========================
# TRAIN / VALIDATION SPLIT
# =========================
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# =========================
# DATALOADERS
# =========================
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

# =========================
# LOAD PRETRAINED MODEL
# =========================
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Freeze pretrained layers
for param in model.parameters():
    param.requires_grad = False

# Replace final classification layer
num_features = model.fc.in_features

model.fc = nn.Linear(num_features, num_classes)

model = model.to(device)

# =========================
# LOSS & OPTIMIZER
# =========================
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

# =========================
# TRAINING SETTINGS
# =========================
epochs = 10

# =========================
# HISTORY STORAGE
# =========================
history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": []
}

best_accuracy = 0.0

# =========================
# TRAINING LOOP
# =========================
for epoch in range(epochs):

    # =====================
    # TRAINING
    # =====================
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    # =====================
    # VALIDATION
    # =====================
    model.eval()

    val_loss_total = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss_total += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_loss = val_loss_total / len(val_loader)
    val_accuracy = 100 * val_correct / val_total

    # =====================
    # SAVE HISTORY
    # =====================
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_accuracy"].append(val_accuracy)

    # =====================
    # SAVE BEST MODEL
    # =====================
    if val_accuracy > best_accuracy:

        best_accuracy = val_accuracy

        os.makedirs("models", exist_ok=True)

        torch.save(
            model.state_dict(),
            "models/best_model.pth"
        )

        print("Best model saved!")

    # =====================
    # PRINT EPOCH RESULTS
    # =====================
    print(f"\nEpoch [{epoch+1}/{epochs}]")
    print(f"Train Loss     : {train_loss:.4f}")
    print(f"Train Accuracy : {train_accuracy:.2f}%")
    print(f"Val Loss       : {val_loss:.4f}")
    print(f"Val Accuracy   : {val_accuracy:.2f}%")

# =========================
# SAVE FINAL MODEL
# =========================
torch.save(
    model.state_dict(),
    "models/final_model.pth"
)

# =========================
# SAVE CLASS NAMES
# =========================
with open("models/class_names.json", "w") as f:
    json.dump(class_names, f)

# =========================
# SAVE TRAINING HISTORY
# =========================
with open("models/history.json", "w") as f:
    json.dump(history, f)

print("\nTraining Completed Successfully!")
print(f"Best Validation Accuracy: {best_accuracy:.2f}%")
print("Model and history saved successfully!")


Using Device: cuda

Classes Found:
0: CCI Leaflet Disease
1: Caterpillar Damage
2: Drying of Leaflets
3: Flaccidity Disease
4: Healthy Leaves
5: Yellowing Disease
Best model saved!

Epoch [1/10]
Train Loss     : 0.6845
Train Accuracy : 79.24%
Val Loss       : 0.3031
Val Accuracy   : 92.24%
Best model saved!

Epoch [2/10]
Train Loss     : 0.2839
Train Accuracy : 92.14%
Val Loss       : 0.1824
Val Accuracy   : 95.51%

Epoch [3/10]
Train Loss     : 0.2231
Train Accuracy : 93.88%
Val Loss       : 0.1531
Val Accuracy   : 95.51%
Best model saved!

Epoch [4/10]
Train Loss     : 0.1831
Train Accuracy : 94.63%
Val Loss       : 0.1243
Val Accuracy   : 96.57%

Epoch [5/10]
Train Loss     : 0.1633
Train Accuracy : 95.10%
Val Loss       : 0.1205
Val Accuracy   : 96.24%

Epoch [6/10]
Train Loss     : 0.1578
Train Accuracy : 94.96%
Val Loss       : 0.1271
Val Accuracy   : 95.76%
Best model saved!

Epoch [7/10]
Train Loss     : 0.1320
Train Accuracy : 95.79%
Val Loss       : 0.0964
Val Accuracy   : 9